#  Online Course Enrollments - PySpark Exercise Set (with Azure Blob Storage)

In [0]:
# 🚀 Initialize Spark Session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("OnlineCourseEnrollments").getOrCreate()


In [0]:
# 🔗 Mount Azure Blob Storage to DBFS
# Replace with your actual storage key
storage_account_name = "subramani21"
storage_account_key = "FGOU2xJ7nBFGsKe8YW2VhNQxWWYmTF6Ez+Q5szV1x3V4BOW/ey9f+D1f4AQ8l1kwahHknbp8rv6e+AStTE4HyQ=="
container_name = "images"

# Mount container to DBFS
dbutils.fs.mount(
  source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
  mount_point = f"/mnt/{container_name}",
  extra_configs = {f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_key}
)


True

In [0]:
# 📂 Verify Mounted Files
display(dbutils.fs.ls("/mnt/images"))


path,name,size,modificationTime
dbfs:/mnt/images/course_details.csv,course_details.csv,177,1750049223000
dbfs:/mnt/images/course_enrollments.csv,course_enrollments.csv,501,1750049211000
dbfs:/mnt/images/customers.csv,customers.csv,130,1749788700000
dbfs:/mnt/images/orders_.csv,orders_.csv,224,1749788737000
dbfs:/mnt/images/output_parquet/,output_parquet/,0,1749789347000
dbfs:/mnt/images/sales_data.csv,sales_data.csv,334,1749724704000


In [0]:
# 📥 Load Course Enrollments CSV
df = spark.read.option("header", True).option("inferSchema", True).csv("/mnt/images/course_enrollments.csv")
df.show()
df.printSchema()


+------------+-----------+--------------------+-----------+----------+---------------+------+---------+
|EnrollmentID|StudentName|          CourseName|   Category|EnrollDate|ProgressPercent|Rating|   Status|
+------------+-----------+--------------------+-----------+----------+---------------+------+---------+
|      ENR001|     Aditya|Python for Beginners|Programming|2024-05-10|             80|   4.5|   Active|
|      ENR002|     Simran|Data Analysis wit...|  Analytics|2024-05-12|            100|   4.7|Completed|
|      ENR003|     Aakash| Power BI Essentials|  Analytics|2024-05-13|             30|   3.8|   Active|
|      ENR004|       Neha|         Java Basics|Programming|2024-05-15|              0|  NULL| Inactive|
|      ENR005|       Zara|Machine Learning 101|         AI|2024-05-17|             60|   4.2|   Active|
|      ENR006|    Ibrahim|Python for Beginners|Programming|2024-05-18|             90|   4.6|Completed|
+------------+-----------+--------------------+-----------+-----

In [0]:
# 🛠 Load Data with Manual Schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("EnrollmentID", StringType(), True),
    StructField("StudentName", StringType(), True),
    StructField("CourseName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("EnrollDate", DateType(), True),
    StructField("ProgressPercent", IntegerType(), True),
    StructField("Rating", DoubleType(), True),
    StructField("Status", StringType(), True)
])

df_manual = spark.read.option("header", True).schema(schema).csv("/mnt/images/course_enrollments.csv")
df_manual.printSchema()


root
 |-- EnrollmentID: string (nullable = true)
 |-- StudentName: string (nullable = true)
 |-- CourseName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- EnrollDate: date (nullable = true)
 |-- ProgressPercent: integer (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
# 🔍 Filter: Students with Progress < 50%
df.filter(df.ProgressPercent < 50).show()


+------------+-----------+-------------------+-----------+----------+---------------+------+--------+
|EnrollmentID|StudentName|         CourseName|   Category|EnrollDate|ProgressPercent|Rating|  Status|
+------------+-----------+-------------------+-----------+----------+---------------+------+--------+
|      ENR003|     Aakash|Power BI Essentials|  Analytics|2024-05-13|             30|   3.8|  Active|
|      ENR004|       Neha|        Java Basics|Programming|2024-05-15|              0|  NULL|Inactive|
+------------+-----------+-------------------+-----------+----------+---------------+------+--------+



In [0]:
# 🧹 Replace Null Ratings with Average
from pyspark.sql.functions import avg, when, col

avg_rating = df.select(avg("Rating")).first()[0]
df = df.withColumn("Rating", when(col("Rating").isNull(), avg_rating).otherwise(col("Rating")))


In [0]:
# Add 'IsActive' Column
df = df.withColumn("IsActive", when(col("Status") == "Active", 1).otherwise(0))


In [0]:
# 📊 Aggregations
df.groupBy("CourseName").agg(avg("ProgressPercent").alias("AvgProgress")).show()
df.groupBy("Category").count().show()

from pyspark.sql.functions import count
df.groupBy("CourseName").agg(count("*").alias("TotalEnrollments")).orderBy(col("TotalEnrollments").desc()).limit(1).show()


+--------------------+-----------+
|          CourseName|AvgProgress|
+--------------------+-----------+
|Data Analysis wit...|      100.0|
|         Java Basics|        0.0|
|Machine Learning 101|       60.0|
|Python for Beginners|       85.0|
| Power BI Essentials|       30.0|
+--------------------+-----------+

+-----------+-----+
|   Category|count|
+-----------+-----+
|Programming|    3|
|         AI|    1|
|  Analytics|    2|
+-----------+-----+

+--------------------+----------------+
|          CourseName|TotalEnrollments|
+--------------------+----------------+
|Python for Beginners|               2|
+--------------------+----------------+



In [0]:
# 🔗 Join with Course Details
details_df = spark.read.option("header", True).option("inferSchema", True).csv("/mnt/images/course_details.csv")
df_joined = df.join(details_df, on="CourseName", how="left")
df_joined.show()


+--------------------+------------+-----------+-----------+----------+---------------+------+---------+--------+-------------+----------+
|          CourseName|EnrollmentID|StudentName|   Category|EnrollDate|ProgressPercent|Rating|   Status|IsActive|DurationWeeks|Instructor|
+--------------------+------------+-----------+-----------+----------+---------------+------+---------+--------+-------------+----------+
|Python for Beginners|      ENR001|     Aditya|Programming|2024-05-10|             80|   4.5|   Active|       1|            4|    Rakesh|
|Data Analysis wit...|      ENR002|     Simran|  Analytics|2024-05-12|            100|   4.7|Completed|       0|            3|    Anjali|
| Power BI Essentials|      ENR003|     Aakash|  Analytics|2024-05-13|             30|   3.8|   Active|       1|            5|     Rekha|
|         Java Basics|      ENR004|       Neha|Programming|2024-05-15|              0|  NULL| Inactive|       0|            6|     Manoj|
|Machine Learning 101|      ENR005

In [0]:
#  Load course_details.csv from mounted Azure Blob path
details_df = spark.read.option("header", True).option("inferSchema", True).csv("/mnt/images/course_details.csv")

details_df.show()


+--------------------+-------------+----------+
|          CourseName|DurationWeeks|Instructor|
+--------------------+-------------+----------+
|Python for Beginners|            4|    Rakesh|
|Data Analysis wit...|            3|    Anjali|
| Power BI Essentials|            5|     Rekha|
|         Java Basics|            6|     Manoj|
|Machine Learning 101|            8|     Samir|
+--------------------+-------------+----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

# 🪟 Define window spec to rank students by progress within each course
window_spec = Window.partitionBy("CourseName").orderBy(col("ProgressPercent").desc())

# Apply rank function
df_ranked = df.withColumn("Rank", rank().over(window_spec))

# Show results
df_ranked.select("StudentName", "CourseName", "ProgressPercent", "Rank").show()


+-----------+--------------------+---------------+----+
|StudentName|          CourseName|ProgressPercent|Rank|
+-----------+--------------------+---------------+----+
|     Simran|Data Analysis wit...|            100|   1|
|       Neha|         Java Basics|              0|   1|
|       Zara|Machine Learning 101|             60|   1|
|     Aakash| Power BI Essentials|             30|   1|
|    Ibrahim|Python for Beginners|             90|   1|
|     Aditya|Python for Beginners|             80|   2|
+-----------+--------------------+---------------+----+



In [0]:
#  Lead & Lag Enroll Dates by Category
from pyspark.sql.functions import lead, lag

window_spec_cat = Window.partitionBy("Category").orderBy("EnrollDate")
df = df.withColumn("NextEnrollDate", lead("EnrollDate").over(window_spec_cat)) \
       .withColumn("PrevEnrollDate", lag("EnrollDate").over(window_spec_cat))
df.select("StudentName", "Category", "EnrollDate", "PrevEnrollDate", "NextEnrollDate").show()


+-----------+-----------+----------+--------------+--------------+
|StudentName|   Category|EnrollDate|PrevEnrollDate|NextEnrollDate|
+-----------+-----------+----------+--------------+--------------+
|       Zara|         AI|2024-05-17|          NULL|          NULL|
|     Simran|  Analytics|2024-05-12|          NULL|    2024-05-13|
|     Aakash|  Analytics|2024-05-13|    2024-05-12|          NULL|
|     Aditya|Programming|2024-05-10|          NULL|    2024-05-15|
|       Neha|Programming|2024-05-15|    2024-05-10|    2024-05-18|
|    Ibrahim|Programming|2024-05-18|    2024-05-15|          NULL|
+-----------+-----------+----------+--------------+--------------+



In [0]:
#  Pivot: Enrollments by Category and Status
df.groupBy("Category").pivot("Status").count().show()


+-----------+------+---------+--------+
|   Category|Active|Completed|Inactive|
+-----------+------+---------+--------+
|Programming|     1|        1|       1|
|         AI|     1|     NULL|    NULL|
|  Analytics|     1|        1|    NULL|
+-----------+------+---------+--------+



In [0]:
#  Extract Year & Month from EnrollDate
from pyspark.sql.functions import year, month
df = df.withColumn("EnrollYear", year("EnrollDate")).withColumn("EnrollMonth", month("EnrollDate"))
df.select("StudentName", "EnrollDate", "EnrollYear", "EnrollMonth").show()


+-----------+----------+----------+-----------+
|StudentName|EnrollDate|EnrollYear|EnrollMonth|
+-----------+----------+----------+-----------+
|     Aditya|2024-05-10|      2024|          5|
|     Simran|2024-05-12|      2024|          5|
|     Aakash|2024-05-13|      2024|          5|
|       Neha|2024-05-15|      2024|          5|
|       Zara|2024-05-17|      2024|          5|
|    Ibrahim|2024-05-18|      2024|          5|
+-----------+----------+----------+-----------+



In [0]:
#  Clean Null Status & Remove Duplicates
df_clean = df.filter((col("Status").isNotNull()) & (col("Status") != ""))
df_clean = df_clean.dropDuplicates(["EnrollmentID"])


In [0]:
#  Export Final Cleaned Data
df_clean.write.mode("overwrite").option("header", True).csv("/mnt/images/output_csv")
df_clean.write.mode("overwrite").json("/mnt/images/output_json")
df_clean.write.mode("overwrite").option("compression", "snappy").parquet("/mnt/images/output_parquet")
